#Customer Sign-Up Behaviour & Data Quality Audit

This notebook analyses customer sign-up data to assess dataset quality issues and explore user acquisition trends. The insights from this analysis support business decisions for marketing and onboarding teams.

## Project Context

Rapid Scale is a fast-growing SaaS company offering tiered subscription plans.  
The Business Intelligence team uses customer sign-up data to support monthly business reviews.

This analysis focuses on:
- data quality and completeness
- customer acquisition sources
- plan selection and marketing opt-in behaviour

In [ ]:
#importing libraries
import pandas as pd
import numpy as np

##Data Loading

The customer sign-up dataset is loaded and inspected to understand its structure, data types, and any potential quality issues such as missing or inconsistent values.

In [ ]:
#loading data
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/customer_signups.csv')


In [ ]:
df.head() #data structure

,customer_id,name,email,signup_date,source,region,plan_selected,marketing_opt_in,age,gender
0,CUST00000,Joshua Bryant,NaN,NaN,Instagram,NaN,basic,No,34,Female
1,CUST00001,Nicole Stewart,nicole1@example.com,02-01-24,LinkedIn,West,basic,Yes,29,Male
2,CUST00002,Rachel Allen,rachel2@example.com,03-01-24,Google,North,PREMIUM,Yes,34,Non-Binary
3,CUST00003,Zachary Sanchez,zachary3@mailhub.org,04-01-24,YouTube,NaN,Pro,No,40,Male
4,CUST00004,NaN,matthew4@mailhub.org,05-01-24,LinkedIn,West,Premium,No,25,Other


##Initial Data Inspection

We review the dataset shape, column names, data types, and sample rows to identify:
- missing values
- incorrect data types
- inconsistent text formatting

In [ ]:
df.info() #check column structure and data types

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   customer_id       298 non-null    object
 1   name              291 non-null    object
 2   email             266 non-null    object
 3   signup_date       298 non-null    object
 4   source            291 non-null    object
 5   region            270 non-null    object
 6   plan_selected     292 non-null    object
 7   marketing_opt_in  290 non-null    object
 8   age               288 non-null    object
 9   gender            292 non-null    object
dtypes: object(10)
memory usage: 23.6+ KB


In [ ]:
df.isna().sum() #checks for missing values

,0
customer_id,2
name,9
email,34
signup_date,2
source,9
region,30
plan_selected,8
marketing_opt_in,10
age,12
gender,8


##Data Cleaning

This step focuses on improving data quality by:
- converting signup_date to datetime format
- standardising inconsistent text values (e.g. plan_selected, gender)
- removing duplicate customer records
- handling missing values where appropriate

In [ ]:
#data cleaning
df['signup_date'] = pd.to_datetime(df['signup_date'], errors='coerce') #convert signup to datetime
df.info() #verifying the datatype conversion


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   customer_id       298 non-null    object        
 1   name              291 non-null    object        
 2   email             266 non-null    object        
 3   signup_date       294 non-null    datetime64[ns]
 4   source            291 non-null    object        
 5   region            270 non-null    object        
 6   plan_selected     292 non-null    object        
 7   marketing_opt_in  290 non-null    object        
 8   age               288 non-null    object        
 9   gender            292 non-null    object        
dtypes: datetime64[ns](1), object(9)
memory usage: 23.6+ KB


/tmp/ipython-input-1628402821.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['signup_date'] = pd.to_datetime(df['signup_date'], errors='coerce') #convert signup to datetime


In [ ]:
#shows unique values and inconsistency in text (object)
df['plan_selected'].value_counts(dropna=False) #checks unique values
df['gender'].value_counts(dropna=False)
df['region'].value_counts(dropna=False)
df['source'].value_counts(dropna=False)
df['marketing_opt_in'].value_counts(dropna=False)


,count
marketing_opt_in,
No,156
Yes,133
NaN,10
Nil,1


In [ ]:
#standardise inconsistent text values
df['plan_selected'] = (
    df['plan_selected']
    .str.strip() #removes spaces before/after text
    .str.lower() #converts to lowercase
    .replace({
        'basic':'Basic',
        'pro':'Pro',
        'premium':'Premium'
        })
    .fillna('Unknown')  #handle missing
)
df['plan_selected'] = df['plan_selected'].where(
    df['plan_selected'].isin(['Basic', 'Pro', 'Premium']),
    'Unknown'
)

df['gender'] = (
    df['gender']
    .astype(str) #fixes numeric values, every value becomes text
    .str.strip()
    .str.title() #capitalise the first letter
    .replace('Nan', 'Unknown') #handles 'nan' created by astype(str)
)
df['gender'] = df['gender'].where(
    df['gender'].isin(['Male', 'Female', 'Non-Binary', 'Other', 'Unknown']),
    'Unknown'
)

df['region'] = (
    df['region']
    .str.strip()
    .str.title()
    .fillna('Unknown')
)

df['source'] = df['source'].str.strip().str.title()

df['marketing_opt_in'] = (
    df['marketing_opt_in']
    .replace('Nil', 'No')
    .str.strip()
    .str.title()
    .fillna('Unknown')
)

In [ ]:
#check for number of duplicates
df.duplicated(subset=['customer_id']).sum()


np.int64(1)

In [ ]:
#drop duplicates/remove duplicate row from customer_id
df=df.drop_duplicates(subset=['customer_id'])
#verifying duplicates
df.duplicated(subset='customer_id').sum()

np.int64(0)

In [ ]:
#handle missing values
df.isna().sum() #check for missing values

# Age values are left as null to avoid introducing bias
# Email missing values retained as they do not affect behavioural analysis
# Region missing values were previously handled using an 'Unknown' category


,0
customer_id,1
name,9
email,34
signup_date,6
source,9
region,0
plan_selected,0
marketing_opt_in,0
age,12
gender,0


##Data Quality Summary

After cleaning, we summarise:
- missing values per column
- percentage of missing data
- number of duplicate records removed
- categories that were standardised for consistency

In [ ]:
#data quality summary
#count of missing values per columns
df.isna().sum()
#% of missing values
(df.isna().sum() / len(df)) * 100
#duplicates were identified and removed earlier using customer_id

#inconsistent categorical values corrected earlier:
  #plan_selected: PRO / pro → Pro
  #gender: male / MALE → Male
  #marketing_opt_in: Nil → No
  #region and source: inconsistent casing standardised

,0
customer_id,0.334448
name,3.010033
email,11.371237
signup_date,2.006689
source,3.010033
region,0.000000
plan_selected,0.000000
marketing_opt_in,0.000000
age,4.013378
gender,0.000000


##Summary Statistics and Aggregations

We use groupby and value_counts to explore:
- weekly sign-up trends
- sign-ups by acquisition source, region, and plan
- marketing opt-in behaviour by gender
- age distribution statistics

In [ ]:
#summary outputs (using pandas aggregation)
#sign-ups per week
df.groupby(df['signup_date'].dt.to_period('W')).size() #groups singup_date into weeks and counts number of sign-ups per week


,0
signup_date,
2024-01-01/2024-01-07,6
2024-01-08/2024-01-14,5
2024-01-15/2024-01-21,7
2024-01-22/2024-01-28,7
2024-01-29/2024-02-04,8
2024-02-05/2024-02-11,6
2024-02-12/2024-02-18,6
2024-02-19/2024-02-25,7
2024-02-26/2024-03-03,7


In [ ]:
#sign-ups by source
df['source'].value_counts() #counts sign-ups per source

,count
source,
Youtube,58
Google,50
Referral,49
Instagram,48
Facebook,40
Linkedin,39
??,6


In [ ]:
#sign-ups by region
df['region'].value_counts() #shows sign-ups by region

,count
region,
North,65
East,61
South,59
West,45
Central,39
Unknown,30


In [ ]:
#sign-ups by plan_selected
df['plan_selected'].value_counts() #counts how many users chose each plan

,count
plan_selected,
Premium,99
Pro,94
Basic,92
Unknown,14


In [ ]:
#marketing opt-in counts by gender
df.groupby('gender')['marketing_opt_in'].value_counts() #counts opt-ins per gender

gender      marketing_opt_in
Female      No                  47
            Yes                 44
            Unknown              1
Male        No                  51
            Yes                 38
            Unknown              3
Non-Binary  No                  20
            Yes                 19
            Unknown              3
Other       No                  32
            Yes                 24
            Unknown              3
Unknown     No                   7
            Yes                  7
Name: count, dtype: int64

In [ ]:
#age summary
df['age'] = pd.to_numeric(df['age'], errors='coerce') #convert age from text to numerical values
df['age'].describe() #statistics

,age
count,280.00000
mean,36.17500
std,14.99802
min,21.00000
25%,25.00000
50%,34.00000
75%,40.00000
max,206.00000


In [ ]:
df['age'].isna().sum() #missing (null) age values in dataset

np.int64(19)

##Stretch Tasks (Optional Analysis)
Additional analysis was completed to demonstrate advanced techniques, including:

- Load the support_tickets.csv dataset
- Join it to customer_signups.csv on customer_id
- Count how many customers contacted support within 2 weeks of sign-up
- Summarise support activity by plan and region (Group by plan and region)

These tasks extend the core analysis but are not required for the main report findings.

In [ ]:
#suport ticket analysis - stretch task
supportticket_df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/support_tickets.csv')
supportticket_df.head()

,ticket_id,customer_id,ticket_date,issue_type,resolved
0,TKT0000-1,CUST00203,2024-08-17,Billing,Yes
1,TKT0000-2,CUST00203,2024-07-22,Technical Error,Yes
2,TKT0000-3,CUST00203,2024-07-22,Other,Yes
3,TKT0001-1,CUST00266,2024-09-26,Account Setup,Yes
4,TKT0001-2,CUST00266,2024-10-09,Technical Error,No


In [ ]:
supportticket_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 123 entries, 0 to 122
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   ticket_id    123 non-null    object
 1   customer_id  123 non-null    object
 2   ticket_date  123 non-null    object
 3   issue_type   123 non-null    object
 4   resolved     123 non-null    object
dtypes: object(5)
memory usage: 4.9+ KB


In [ ]:
#convert ticket_date to datetime datatype
supportticket_df['ticket_date'] = pd.to_datetime(supportticket_df['ticket_date'], errors='coerce')
supportticket_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 123 entries, 0 to 122
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   ticket_id    123 non-null    object        
 1   customer_id  123 non-null    object        
 2   ticket_date  123 non-null    datetime64[ns]
 3   issue_type   123 non-null    object        
 4   resolved     123 non-null    object        
dtypes: datetime64[ns](1), object(4)
memory usage: 4.9+ KB


In [ ]:
#joining customer_signups.csv on customer_id
joined_df=pd.merge(df, supportticket_df, on='customer_id', how='inner') #joins signs-ups and support tickets using customer_id, keeping customers that appear in both datasets
joined_df.head()

,customer_id,name,email,signup_date,source,region,plan_selected,marketing_opt_in,age,gender,ticket_id,ticket_date,issue_type,resolved
0,CUST00005,John Gonzales,john5@mailhub.org,2024-06-01,Facebook,South,Premium,No,34.0,Other,TKT0008-1,2024-06-04,Other,Yes
1,CUST00007,Michael Bailey,michael7@mailhub.org,2024-08-01,Youtube,Central,Pro,Yes,60.0,Other,TKT0036-1,2024-08-07,Billing,Yes
2,CUST00007,Michael Bailey,michael7@mailhub.org,2024-08-01,Youtube,Central,Pro,Yes,60.0,Other,TKT0036-2,2024-08-23,Other,Yes
3,CUST00009,Cindy Anderson,NaN,2024-10-01,Google,East,Premium,No,29.0,Female,TKT0003-1,2024-10-03,Technical Error,No
4,CUST00017,Patty Paul,patty17@inboxmail.net,2024-01-18,Youtube,East,Pro,No,53.0,Non-Binary,TKT0030-1,2024-02-03,Other,Yes


In [ ]:
#how many customers contacted support within 2 weeks (14 days) of sign-up
support_within_2_weeks = joined_df[
    (joined_df['ticket_date'] >= joined_df['signup_date']) & #keep customers that contacted support after sign up
    (joined_df['ticket_date'] <= joined_df['signup_date'] + pd.Timedelta(days=14)) #within 2 weeks
]

support_within_2_weeks['customer_id'].nunique() #counts unique customers


47

In [ ]:
#support activity by plan and region
support_activity=(
    support_within_2_weeks #uses only customers who contacted support within 2 weeks of sign-up
    .groupby(['plan_selected', 'region']) #group support contacts by plan and region
    .size() #counts support contacts in each group
    .sort_values(ascending=False) #showest highest support activity first
)

support_activity

plan_selected  region 
Basic          South      9
Pro            North      7
Premium        West       6
Basic          East       6
Pro            East       6
               Central    4
Premium        Central    4
Basic          West       4
Unknown        North      4
Premium        North      3
Pro            West       3
               South      3
Basic          Central    2
               North      2
Premium        South      1
               East       1
Basic          Unknown    1
Unknown        East       1
dtype: int64